In [ ]:
# input
pred_file = "./data/pred_ge_3_clique_3.tsv"
# output
site_plddt_file = "./data/pred_site_plddt_proba.tsv"

In [2]:
import pandas as pd
df = pd.read_table(pred_file)

In [ ]:
from itertools import combinations
import tqdm
import networkx as nx

def get_site(site_str: str):
    g = nx.Graph()
    sites = site_str.split(";")
    for s in sites:
        members = s.split(",")
        for (i, j) in combinations(members, 2):
            g.add_edge(i, j)
    
    result: list[list[str]] = []
    for s in nx.connected_components(g): # viewed as a site
        result.append(sorted(list(s)))
    return result


records = []
for _, row in tqdm.tqdm(df.iterrows(), total=len(df)):
    posi_to_plddt = dict(
        zip(
            row['posi'].split(","),
            row['plddt'].split(",")
        )
    )

    posi_to_proba = dict(
        zip(
            row['posi'].split(","),
            row['pred'].split(",")
        )
    )
    
    g = nx.Graph()
    sites = get_site(row['site']) # row['site'] is actually cliques and may have residue intersection; now we defined sites to avoid intersection
    site_keys = []
    site_plddts = []
    site_probas = []
    for s in sites:
        positions = s
        plddts = [posi_to_plddt[i] for i in positions]
        probas = [posi_to_proba[i] for i in positions]
        site_keys.append(",".join(positions))
        site_plddts.append(",".join(plddts))
        site_probas.append(",".join(probas))
    
    seq_id = row['seq_id'].split("-")[1]
    for i in range(len(site_keys)):
        records.append({
            "seq_id": seq_id,
            "site": site_keys[i],
            "plddt": site_plddts[i],
            "proba": site_probas[i],
        })

# 1 hour
pd.DataFrame(records).to_csv(site_plddt_file, sep="\t", index=None)

100%|██████████| 38361041/38361041 [49:41<00:00, 12868.18it/s] 
